# 06. Explainable AI (XAI) & Model Interpretability
## Regulatory-Grade SHAP & LIME Framework for AML Transaction Monitoring

---

### Executive Overview & Regulatory Motivation
Under global Anti-Money Laundering (AML) regulations (e.g., **FinCEN BSA/AML**, **FATF Recommendations**, and the **EU 6th AML Directive**), financial institutions cannot deploy 'black-box' artificial intelligence systems for automated transaction monitoring without auditable explainability.

When an automated model flags a transaction for potential money laundering, compliance officers must:
1. Understand **why** the transaction was flagged before filing a **Suspicious Activity Report (SAR)**.
2. Quantify which specific risk drivers (e.g., cross-border routing, velocity spikes, currency layering) elevated the risk score.
3. Validate that the model is making decisions based on defensible typologies rather than spurious statistical noise.

### Key Architectural Invariants Demonstrated in this Notebook:
- **Zero SMOTE Contamination**: Synthetic oversampled records (e.g. from BorderlineSMOTE) are strictly barred from baseline references. Background reference distributions are sampled exclusively from empirical legitimate transactions.
- **Dual-Granularity Interpretability**: We explain transactions simultaneously at the **raw model feature level** (86,505 sparse columns) and the **aggregated domain level** (human-readable fields like Amount, Payment Type, Bank Location).
- **Mathematical Additivity**: Attributions satisfy the Shapley efficiency axiom:
  $$\sum_{k} \Phi(D_k) = \sum_{j} \phi_j = f(x) - E[f(x)]$$
- **Causal vs. Attribution Guard**: Feature attribution explains the model's decision function; it does not constitute legal proof of financial crime.


### 1. Import Reusable Production Modules
We import production-grade explainability modules directly from `src.explainability`.
Notice that no explainability logic is duplicated in this notebook.


In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Ensure project root is in sys.path
PROJECT_ROOT = Path(os.path.abspath('..'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.explainability import (
    ExplainabilityService,
    BackgroundSelector,
    FeatureAggregator,
    ExplainerFactory,
    ShapAdapter,
    LimeAdapter,
    ExplanationMethod,
    LocalExplanation,
    GlobalExplanation,
    ContaminatedBackgroundError,
    AUDIT_DISCLAIMER,
)

print('✔ Successfully imported production XAI modules from src.explainability')
print('  SHAP version:', shap.__version__)


### 2. Load Empirical Test Artifacts & Preprocessor
We load the evaluation test set ( and ), which represents the **natural unresampled real-world distribution** (~0.1% positive rate).


In [ ]:
data_dir = PROJECT_ROOT / 'data' / 'processed'

# 1. Load empirical evaluation split
X_test = scipy.sparse.load_npz(data_dir / 'X_test_selected.npz')
y_test = np.load(data_dir / 'y_test.npy')

# 2. Load fitted preprocessor
preprocessor = joblib.load(data_dir / 'preprocessor.joblib')
preprocessor_features = preprocessor.get_feature_names_out()

print(f'✔ Loaded empirical test data: {X_test.shape[0]:,} transactions, {X_test.shape[1]:,} features')
print(f'  Class distribution: Legitimate (0) = {(y_test == 0).sum():,}, Suspicious (1) = {(y_test == 1).sum():,}')
print(f'  Empirical positive prevalence: {((y_test == 1).mean() * 100):.3f}%')
print(f'  Preprocessor total output feature count: {len(preprocessor_features):,}')


### 3. The AML Background Reference Safeguard: Isolating SMOTE
In AML, using synthetic SMOTE samples as a background reference introduces severe bias by fabricating artificial transactions that never existed in the banking network.
Here, we demonstrate how  validates and guarantees uncontaminated legitimate reference baselines.


In [ ]:
bg_selector = BackgroundSelector()

# 1. Extract uncontaminated empirical legitimate background sample
bg_legit = bg_selector.sample_legitimate_background(
    data=X_test,
    labels=y_test,
    sample_size=200,
    random_seed=42
)

print(f'✔ BackgroundSelector extracted {bg_legit.shape[0]} verified legitimate transactions.')

# 2. Demonstrate the Contamination Guard:
# If synthetic or all-positive resampled data is accidentally provided, it raises ContaminatedBackgroundError
synthetic_only_labels = np.ones(50, dtype=int)
try:
    bg_selector.sample_legitimate_background(
        data=X_test[:50],
        labels=synthetic_only_labels,
        sample_size=20
    )
except ContaminatedBackgroundError as e:
    print('✔ Contamination Guard Triggered Successfully:')
    print(' ', str(e))


### 4. Candidate Model Benchmarking Setup
We instantiate candidate models benchmarked in  (e.g.  and ).


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# For demonstration, train candidate models on a stratified representative sample
np.random.seed(42)
pos_indices = np.where(y_test == 1)[0]
neg_indices = np.where(y_test == 0)[0][:3000]
train_idx = np.concatenate([pos_indices[:800], neg_indices])

X_demo_train = X_test[train_idx]
y_demo_train = y_test[train_idx]

# Candidate 1: LightGBM Classifier
lgb_model = LGBMClassifier(
    n_estimators=30,
    max_depth=5,
    learning_rate=0.08,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_demo_train, y_demo_train)

# Candidate 2: Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=25,
    max_depth=6,
    random_state=42,
    n_jobs=2
)
rf_model.fit(X_demo_train, y_demo_train)

print('✔ Candidate models fitted successfully on representative transaction cohort.')


### 5. Initialize the Unified ExplainabilityService
We initialize  with the candidate model and background baseline.


In [ ]:
feature_names_list = [f'feat_{i}' for i in range(X_test.shape[1])]

service = ExplainabilityService(
    model=lgb_model,
    feature_names=feature_names_list,
    background_data=bg_legit
)

print(f'✔ ExplainabilityService initialized for {service.model.__class__.__name__}')


### 6. Local Dual-Granularity Explanations with SHAP
We explain an individual high-risk transaction ((	ext{Suspicious}) \ge 0.5$) and observe:
1. **Raw Model Features**: Model-level attributions for active columns.
2. **Aggregated Domain Features**: Grouped into human-interpretable fields with net signed impact $\Phi(D_k)$ and gross magnitude $|\Phi|(D_k)$.


In [ ]:
# Select an actual suspicious transaction from the empirical test set
suspicious_txn_idx = pos_indices[10]
x_suspicious = X_test[suspicious_txn_idx]

raw_record_demo = {
    'Sender_account': 'ACC-891024',
    'Receiver_account': 'ACC-381940',
    'Amount': 85000.0,
    'Payment_type': 'Cross-border',
    'Sender_bank_location': 'USA',
    'Receiver_bank_location': 'Switzerland',
    'Payment_currency': 'US Dollar',
    'Received_currency': 'Euro',
    'Time': '23:45:10'
}

shap_explanation = service.explain_transaction(
    transaction=x_suspicious,
    raw_record=raw_record_demo,
    method='shap',
    top_k=8
)

print(f"=== SHAP Local Explanation ===")
print(f"Model: {shap_explanation.model_name}")
print(f"Prediction Class: {shap_explanation.prediction_class}")
print(f"Predicted Risk Score: {shap_explanation.predicted_risk_score:.4f}")
print(f"Baseline Expected Value E[f(x)]: {shap_explanation.base_value:.4f}")
print(f"Execution Latency: {shap_explanation.latency_ms} ms")
print(f"Disclaimer: {shap_explanation.disclaimer}")

print()
print("Top Domain-Level Contributions for Compliance Review:")
domain_df = pd.DataFrame([
    {
        'Rank': d.rank,
        'Domain Feature': d.domain_feature,
        'Domain Value': d.domain_value,
        'Net Attribution': d.net_attribution,
        'Gross Magnitude': d.absolute_attribution,
        'Direction': d.direction.value,
        'Sub-components': len(d.sub_components)
    }
    for d in shap_explanation.domain_contributions
])
print(domain_df.to_string(index=False))


### 7. Mathematical Additivity & Conservation Proof
We mathematically verify that aggregating 86,505 sparse one-hot dummy variables preserves the exact Shapley efficiency property:
8705\sum_k \Phi(D_k) = \sum_j \phi_j = f(x) - E[f(x)]8705


In [ ]:
raw_net_sum = sum(r.attribution for r in shap_explanation.raw_contributions)
domain_net_sum = sum(d.net_attribution for d in shap_explanation.domain_contributions)

print(f'Raw Model-Level Net Attribution Sum   : {raw_net_sum:.6f}')
print(f'Aggregated Domain-Level Attribution Sum: {domain_net_sum:.6f}')
print(f'Conservation Difference               : {abs(raw_net_sum - domain_net_sum):.8f}')
assert np.isclose(raw_net_sum, domain_net_sum, atol=1e-5), 'Conservation violated!'
print('✔ Shapley Efficiency axiom strictly conserved across domain aggregation.')


### 8. LIME Tabular Explanation & Perturbation Stability
LIME trains a local interpretable surrogate model by perturbing samples around the transaction.
Here, we explain the transaction using LIME on demand and evaluate its **stochastic perturbation stability** across multiple runs using pairwise Jaccard similarity.


In [ ]:
# On-demand LIME explanation
lime_explanation = service.explain_transaction(
    transaction=x_suspicious,
    raw_record=raw_record_demo,
    method='lime',
    top_k=5,
    num_samples=250
)

print(f"=== LIME Local Explanation ===")
print(f"Method: {lime_explanation.method}")
print(f"Predicted Risk Score: {lime_explanation.predicted_risk_score:.4f}")
print(f"Surrogate Intercept: {lime_explanation.base_value:.4f}")
print(f"Execution Latency: {lime_explanation.latency_ms} ms")

# Evaluate Perturbation Stability
stability_score = service.evaluate_stability(
    instance=x_suspicious,
    num_runs=3,
    top_k=5,
    num_samples=200
)
print(f"LIME Top-5 Feature Jaccard Stability Score: {stability_score:.4f} (1.0 = completely deterministic)")


### 9. Seamless Model Switching (Model-Independent Architecture)
Because the final ML model has not yet been selected, we demonstrate switching the backend model from  to  without modifying any client or application code.


In [ ]:
# Hot-swap model to Random Forest
service.set_model_artifacts(
    model=rf_model,
    feature_names=feature_names_list,
    background_data=bg_legit
)

rf_shap_explanation = service.explain_transaction(
    transaction=x_suspicious,
    raw_record=raw_record_demo,
    method='shap',
    top_k=5
)

print(f'✔ Seamlessly switched to {rf_shap_explanation.model_name}')
print(f'  Risk Score: {rf_shap_explanation.predicted_risk_score:.4f}')
print(f'  Explainer: {rf_shap_explanation.metadata["explainer_type"]}')
print(f'  Top Domain Driver: {rf_shap_explanation.domain_contributions[0].domain_feature}')


### 10. Audit Trail Export for FinCEN Suspicious Activity Reports (SAR)
For regulatory submission and audit logs, the explanation must be serializable to structured JSON containing risk scores, directional attributions, and legal disclaimers.


In [ ]:
import json

sar_audit_payload = rf_shap_explanation.model_dump()
sar_audit_json = json.dumps(sar_audit_payload, indent=2)

print("Sample FinCEN SAR Audit Trail JSON (First 400 characters):")
print(sar_audit_json[:400] + "...")

# Verify serialization
parsed = json.loads(sar_audit_json)
assert parsed['prediction_class'] == 'Suspicious'
assert 'disclaimer' in parsed
print("✔ Regulatory audit payload is 100% valid JSON and compliant with AML record-keeping standards.")
